In [1]:
# PART 1: Imports and Data Load
# ----------------------------
import pandas as pd
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

In [ ]:
# Load and Prepare Dataset
# ----------------------------
df = pd.read_csv("HR_Employee_Attrition.csv")

# Convert Attrition to binary
df["Attrition"] = df["Attrition"].map({"Yes": 1, "No": 0})

# Add Employee ID column
df.insert(0, "EmployeeID", range(1001, 1001 + len(df)))

# Drop irrelevant columns
df.drop(columns=["EmployeeCount", "EmployeeNumber", "Over18", "StandardHours"], inplace=True)

# Confirm data
print("✅ Data Loaded. Sample:")
display(df.head())

✅ Data Loaded. Sample:


,EmployeeID,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EnvironmentSatisfaction,...,PerformanceRating,RelationshipSatisfaction,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,1001,41,1,Travel_Rarely,1102,Sales,1,2,Life Sciences,2,...,3,1,0,8,0,1,6,4,0,5
1,1002,49,0,Travel_Frequently,279,Research & Development,8,1,Life Sciences,3,...,4,4,1,10,3,3,10,7,1,7
2,1003,37,1,Travel_Rarely,1373,Research & Development,2,2,Other,4,...,3,2,0,7,3,3,0,0,0,0
3,1004,33,0,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,4,...,3,3,0,8,3,3,8,7,3,0
4,1005,27,0,Travel_Rarely,591,Research & Development,2,1,Medical,1,...,3,4,1,6,3,3,2,2,2,2


In [3]:
# Load and Prepare Dataset
# ----------------------------
df = pd.read_csv("G:\\Project\\Employee attrition\\WA_Fn-UseC_-HR-Employee-Attrition.csv")

# Convert Attrition to binary
df["Attrition"] = df["Attrition"].map({"Yes": 1, "No": 0})

# Add Employee ID column
df.insert(0, "EmployeeID", range(1001, 1001 + len(df)))

# Drop irrelevant columns
df.drop(columns=["EmployeeCount", "EmployeeNumber", "Over18", "StandardHours"], inplace=True)

# Confirm data
print("✅ Data Loaded. Sample:")
display(df.head())

✅ Data Loaded. Sample:


,EmployeeID,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EnvironmentSatisfaction,...,PerformanceRating,RelationshipSatisfaction,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,1001,41,1,Travel_Rarely,1102,Sales,1,2,Life Sciences,2,...,3,1,0,8,0,1,6,4,0,5
1,1002,49,0,Travel_Frequently,279,Research & Development,8,1,Life Sciences,3,...,4,4,1,10,3,3,10,7,1,7
2,1003,37,1,Travel_Rarely,1373,Research & Development,2,2,Other,4,...,3,2,0,7,3,3,0,0,0,0
3,1004,33,0,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,4,...,3,3,0,8,3,3,8,7,3,0
4,1005,27,0,Travel_Rarely,591,Research & Development,2,1,Medical,1,...,3,4,1,6,3,3,2,2,2,2


In [4]:
# PART 2: Live Model with SMOTE
# ----------------------------

# Step 1: Prepare feature set and target
X_all = df.drop(columns=["Attrition", "EmployeeID"])
y_all = df["Attrition"]

# Step 2: One-hot encode categorical variables
X_all_encoded = pd.get_dummies(X_all, drop_first=True)

# Step 3: Balance the data using SMOTE
smote = SMOTE(random_state=42)
X_balanced, y_balanced = smote.fit_resample(X_all_encoded, y_all)

# Step 4: Train XGBoost on full balanced data
xgb_live = XGBClassifier(
    eval_metric='logloss',
    max_depth=4,
    learning_rate=0.03,
    n_estimators=1000,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.5,
    reg_lambda=1,
    random_state=42
)


xgb_live.fit(X_balanced, y_balanced)

# Step 5: Live prediction function with threshold tuning
def predict_attrition_by_id(emp_id, threshold=0.9):
    row = df[df["EmployeeID"] == emp_id]

    if row.empty:
        print(f"❌ Employee ID {emp_id} not found.")
        return

    row_input = row.drop(columns=["Attrition", "EmployeeID"])
    row_encoded = pd.get_dummies(row_input, drop_first=True)
    row_encoded = row_encoded.reindex(columns=X_all_encoded.columns, fill_value=0)

    prob = xgb_live.predict_proba(row_encoded)[0][1]
    prediction = 1 if prob >= threshold else 0
    status = "Likely to Leave (Attrition = 1)" if prediction == 1 else "Likely to Stay (Attrition = 0)"

    print(f"🧑‍💼 Employee ID {emp_id} → {status} | Confidence: {round(prob, 2)}")
    print("Probabaility of Employee Leaving: ", xgb_live.predict_proba(row_encoded)[0][1])

    explainer = shap.TreeExplainer(xgb_live)
    # print("Base value (average model prediction):", explainer.expected_value)


In [5]:
# ----------------------------
# PART 3: Model Evaluation
# ----------------------------

# Step 1: Train-test split
# X_eval = pd.get_dummies(df.drop(columns=["Attrition", "EmployeeID"]), drop_first=True)
# y_eval = df["Attrition"]

# X_train, X_test, y_train, y_test = train_test_split(
#     X_eval, y_eval, test_size=0.2, stratify=y_eval, random_state=42)
# 

# # Step 2: Apply SMOTE to training data only
# smote = SMOTE(random_state=42)
# X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

# # Step 3: Train model for evaluation

# xgb_live = XGBClassifier(
#     eval_metric='logloss',
#     max_depth=4,
#     learning_rate=0.03,
#     n_estimators=1000,        # more trees for stability
#     subsample=0.8,            # avoid overfitting
#     colsample_bytree=0.8,
#     reg_alpha=0.5,            # L1 regularization
#     reg_lambda=1,             # L2 regularization
#     random_state=42
# )

# xgb_eval.fit(X_train_bal, y_train_bal)

# # Step 4: Predict with threshold tuning
# y_probs = xgb_eval.predict_proba(X_test)[:, 1]
# threshold = 0.9
# y_pred = (y_probs >= threshold).astype(int)

# # Step 5: Evaluation Metrics
# print("✅ XGBoost Evaluation with SMOTE & Threshold Tuning")
# print(f"Threshold: {threshold}")
# print("Accuracy:", accuracy_score(y_test, y_pred))
# print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
# print("\nClassification Report:\n", classification_report(y_test, y_pred))

In [6]:
import shap

# Step 1: Initialize SHAP explainer with trained model
explainer = shap.TreeExplainer(xgb_live)

# Step 2: Prepare full input feature matrix for SHAP
X_for_shap = pd.get_dummies(df.drop(columns=["Attrition", "EmployeeID"]), drop_first=True)
X_for_shap = X_for_shap.reindex(columns=X_all_encoded.columns, fill_value=0)

# Step 3: Get SHAP values for the entire dataset
shap_values = explainer.shap_values(X_for_shap)

# Step 4: Function to explain one employee
def explain_employee_factors(emp_id, top_n=5):
    # Find the row index
    match = df[df["EmployeeID"] == emp_id]
    if match.empty:
        print(f"❌ Employee ID {emp_id} not found.")
        return

    row_idx = match.index[0]
    feature_names = X_for_shap.columns
    shap_row = shap_values[row_idx]
    row_data = X_for_shap.iloc[row_idx]

    # Combine feature names with SHAP impacts and actual values
    impact_df = pd.DataFrame({
        "Feature": feature_names,
        "SHAP_Impact": shap_row,
        "Value": row_data.values
    })

    # Sort by absolute impact (descending), then get top N
    top_factors = impact_df.reindex(
        impact_df["SHAP_Impact"].abs().sort_values(ascending=False).index
    ).head(top_n)

    print(f"\n🔍 Top {top_n} factors influencing attrition prediction for Employee ID {emp_id}:\n")
    print(top_factors[["Feature", "Value", "SHAP_Impact"]].to_string(index=False))


In [10]:
x=int(input(" Enter Employee ID"))
#  Example:
predict_attrition_by_id(x)
explain_employee_factors(x)

 Enter Employee ID 1004


🧑‍💼 Employee ID 1004 → Likely to Stay (Attrition = 0) | Confidence: 0.8899999856948853
Probabaility of Employee Leaving:  0.8924592

🔍 Top 5 factors influencing attrition prediction for Employee ID 1004:

                     Feature Value  SHAP_Impact
  JobRole_Research Scientist  True    -1.162179
                OverTime_Yes  True     1.028353
EducationField_Life Sciences  True    -0.973694
                   DailyRate  1392    -0.913791
            StockOptionLevel     0     0.750171
